### 生データ

In [74]:
import pandas as pd

# 任意のCSVファイルのパスを指定
csv_file_path = "C:\\Users\\funah\\Desktop\\実験関連\\実験データ\\raw_data\\8.all_12_block.csv"
#csv_file_path = "all_12_ID_cleand.csv"

# CSVファイルを読み込み、df_cleaned に格納
df_cleaned = pd.read_csv(csv_file_path)

C:\Users\funah\AppData\Local\Temp\ipykernel_20560\2058740416.py:8: DtypeWarning: Columns (9) have mixed types. Specify dtype option on import or set low_memory=False.
  df_cleaned = pd.read_csv(csv_file_path)











































### ウィンドウサイズに分割

ウインドウサイズとオーバーラップを指定

In [75]:
window_size = 32 # ウィンドウサイズ（秒）
overlap = window_size / 2      # オーバーラップ（秒）
step_size = window_size - overlap  # ステップサイズ（ウィンドウの移動量）
print(step_size)

16.0


前後のデータ削除（8秒以上の問題に関して）（非実行）

In [76]:
# sampling_rate = 60
# Invalid_t = sampling_rate

# # Task_ID が変わるたびに block_id を振る
# #df_cleaned['block_id'] = (df_cleaned['Task_ID'] != df_cleaned['Task_ID'].shift()).cumsum()

# # 各 block_id ごとの長さを計算
# block_lengths = df_cleaned.groupby('block_id').size()

# # 10秒以上のブロックID
# long_blocks = block_lengths[block_lengths >= 10 * sampling_rate].index

# # 切り替え点のインデックス（block_idが変わる所）
# change_indices = df_cleaned[df_cleaned['block_id'].diff().fillna(0) != 0].index

# # 削除対象インデックスを集める
# to_drop = set()

# for idx in change_indices:
#     prev_block_id = df_cleaned.loc[idx - 1, 'block_id'] if idx > 0 else None
#     curr_block_id = df_cleaned.loc[idx, 'block_id']
    
#     # 前のブロックが長い → 終了側1秒削除
#     if prev_block_id in long_blocks:
#         start_idx = max(0, idx - Invalid_t)
#         end_idx = idx - 1
#         to_drop.update(range(start_idx, end_idx + 1))
        
#     # 今のブロックが長い → 開始側1秒削除
#     if curr_block_id in long_blocks:
#         start_idx = idx
#         end_idx = min(len(df_cleaned) - 1, idx + Invalid_t - 1)
#         to_drop.update(range(start_idx, end_idx + 1))

# # データ削除
# df_cleaned = df_cleaned.drop(to_drop).reset_index(drop=True)


データをウインドウサイズで分割する関数 ( 前 vs 後ろ )

In [77]:
def split_into_windows(data, window_size, step_size, align_from_end=False):
    """
    時系列データをウィンドウに分割する。
    
    Parameters:
        data: pandas.DataFrame
        window_size: float (秒)
        step_size: float (秒)
        align_from_end: bool
            False: タスクの前からウィンドウを割り当てる（デフォルト）
            True : タスクの後ろからウィンドウを割り当てる
    """
    windows = []
    skipped_blocks = []

    sampling_rate = 60
    window_size_points = int(window_size * sampling_rate)
    step_size_points = int(step_size * sampling_rate)

    for block_id, block_data in data.groupby('block_id'):
        block_data = block_data.reset_index(drop=True)
        num_samples = len(block_data)

        if num_samples < window_size_points:
            skipped_blocks.append(block_id)
            continue

        if align_from_end:
            starts = list(range(num_samples - window_size_points, -1, -step_size_points))
        else:
            starts = list(range(0, num_samples - window_size_points + 1, step_size_points))

        num_windows = 0
        for start in starts:
            window = block_data.iloc[start : start + window_size_points]
            windows.append(window)
            num_windows += 1

        if num_windows == 0:
            skipped_blocks.append(block_id)

    return windows, skipped_blocks

ウインドウサイズで分割

In [78]:
# データをウィンドウに分割
windows, skipped_blocks = split_into_windows(df_cleaned, window_size, step_size)

# ウィンドウの数を表示
print(f"ウィンドウの数: {len(windows)}")
print(f"スキップされたブロック数: {len(skipped_blocks)}")
print(f"スキップされた block_id: {skipped_blocks}")

# 各ウィンドウのデータサイズを表示
#for i, window in enumerate(windows):
#    print(f"ウィンドウ {i+1} のデータ数: {len(window)}")
#    print(window, "\n")

# 有効なウインドウをCSVファイルに保存
#valid_windows = pd.concat(windows, ignore_index=True)
#valid_windows.to_csv('windows.csv', index=False)

ウィンドウの数: 3777
スキップされたブロック数: 77
スキップされた block_id: [23, 27, 48, 59, 66, 108, 146, 153, 155, 207, 216, 244, 295, 323, 330, 345, 346, 347, 349, 353, 357, 360, 363, 367, 373, 375, 378, 384, 388, 389, 391, 397, 399, 407, 413, 420, 427, 437, 438, 444, 448, 449, 459, 482, 484, 485, 505, 506, 515, 524, 539, 541, 549, 554, 556, 559, 567, 570, 576, 578, 594, 602, 604, 605, 611, 620, 622, 625, 626, 633, 634, 640, 663, 667, 670, 676, 678]


### ウインドウの削除（非実行）

削除

In [90]:
# 有効なウインドウの数を表示
# print(f"削除前のウインドウの数: {len(windows)}")

# # 各ウインドウ内のInvalidの秒数がウインドウサイズの1割以下のウインドウを保持
# windows = [
#     window for window in windows 
#     if len(window) > 0 and window['Validity'].value_counts().get('Invalid', 0) / len(window) <= 0.1
# ]

# # 有効なウインドウの数を表示
# print(f"削除後のウインドウの数: {len(windows)}")

# 各有効なウインドウのデータサイズを表示
#for i, window in enumerate(windows):
#    print(f"有効なウインドウ {i+1} のデータ数: {len(window)}")
#    print(window, "\n")

# 有効なウインドウをCSVファイルに保存
#valid_windows = pd.concat(windows, ignore_index=True)
#valid_windows.to_csv('windows.csv', index=False)

### 文章毎に分割（非実行）

文章毎に分割

In [91]:
# import pandas as pd

# # CSVファイルを読み込み
# df_cleaned = pd.read_csv('C:\\Users\\funah\\Desktop\\実験関連\\実験データ\\予備\\all_12_ID_normalized.csv')

# # windowsという空の配列を定義
# windows = []
# current_window = []  # 現在のウィンドウ

# frag = 0 # フラグ

# # Summary列のデータが存在するところで分割し、直前のグループに追加
# for i, row in df_cleaned.iterrows():
#     if pd.notna(row['Summary']):  # Summary列にデータがあった場合
#         if len(current_window) > 0:  # 現在のウィンドウが空でない場合
#             windows.append(pd.DataFrame(current_window))  # DataFrameとして保存
#         current_window.append(row)  # Summary行を新しいウィンドウに
#         frag = 1
#     elif frag == 1:
#         current_window.append(row)
#         frag = 2
#     elif frag == 2:
#         current_window = [row]
#         frag = 0
#     else:
#         current_window.append(row)  # Summaryがない場合、現在のウィンドウに行を追加


# # 各ウィンドウのデータサイズと内容を表示
# #for i, window in enumerate(windows):
# #    print(f"ウィンドウ {i+1} のデータ数: {len(window)}")
# #    print(pd.DataFrame(window), "\n")

# # 結果を1つのCSVファイルとして保存（任意）
# #combined_data = pd.concat([pd.DataFrame(window) for window in windows], keys=range(1, len(windows) + 1), names=['WindowID'])
# #combined_data.reset_index(level='WindowID', inplace=True)
# #combined_data.to_csv('windows.csv', index=False)

# #print("分割データを 'windows.csv' に保存しました。")

ウィンドウの長さを計算

In [92]:
# # window_size列を追加
# features['window_size'] = None  # 初期値としてNoneを設定

# # 各ウィンドウのサイズを計算して追加
# for i, window in enumerate(windows):
#     if not window.empty:
#         start_time = window['Recording timestamp (s)'].iloc[0]
#         end_time = window['Recording timestamp (s)'].iloc[-1]
#         features.at[i, 'window_size'] = end_time - start_time

# features.to_csv('features_output.csv', index=False)

### 視線特徴量計算

特徴量を入れるデータフレームを作成

In [79]:
import pandas as pd

# 特徴量を格納するためのデータフレームを作成
features = pd.DataFrame(columns=[
    'Participant_ID', 'Gender_ID', 'Task_ID', 'Block_ID', 'Confusion', 'Difficulty',
    'fixation_duration', 'fixation_count', 'fixation_dispersion',
    'saccade_duration', 'saccade_count', 'saccade_amplitude',
    'saccade_velocity', 'relative_saccade_angle', 'absolute_saccade_angle',
    'horizontal_saccade_ratio', 'fixation_saccade_ratio',
    'left_pupil_diameter', 'right_pupil_diameter', 
    'keystroke_interval_mean', 'keystroke_interval_std',
    'keystroke_count', 'average_keystroke_speed', 'keystroke_speed_std',
    'keystroke_max_speed', 'keystroke_min_speed',
    'delete_key_count', 'delete_key_frequency', 
    'sit1', 'sit2', 'sit3', 'sit4', 'sit5', 
    'sit6', 'sit7', 'sit8', 'sit9', 'sit10'
])

# 各ウィンドウに対応する特徴量を追加（NaN で初期化）
features = pd.concat([features, pd.DataFrame(index=range(len(windows)))], ignore_index=True)

# ID, Confusion, Difficulty を設定
features['Participant_ID'] = [w['Participant_ID'].mode()[0] if 'Participant_ID' in w and not w['Participant_ID'].empty else None for w in windows]
features['Gender_ID'] = [w['Gender_ID'].mode()[0] if 'Gender_ID' in w and not w['Gender_ID'].empty else None for w in windows]
features['Task_ID'] = [w['Task_ID'].mode()[0] if 'Task_ID' in w and not w['Task_ID'].empty else None for w in windows]
features['Block_ID'] = [w['block_id'].mode()[0] if 'block_id' in w and not w['block_id'].empty else None for w in windows]
features['Confusion'] = [w['Confusion'].mode()[0] if 'Confusion' in w and not w['Confusion'].empty else None for w in windows]
features['Difficulty'] = [w['Difficulty'].mode()[0] if 'Difficulty' in w and not w['Difficulty'].empty else None for w in windows]


固視時間 [ms] (FD)：各ウィンドウ内のある固視にかかった時間  

In [80]:
# 固視時間の特徴量を計算する関数を定義
def calculate_fixation_duration(df):
    fixation_duration = 0
    is_fixation_active = False
    start_time = None
    
    for _, row in df.iterrows():
        event_type = row['Eye movement type']
        
        if event_type == 'Fixation':
            # 固視が開始した場合、開始時間を記録
            if not is_fixation_active:
                start_time = row['Recording timestamp (s)']
                is_fixation_active = True
                
        elif event_type == 'Saccade' and is_fixation_active:
            # 固視終了時に終了時間を記録し、差分を計算
            end_time = row['Recording timestamp (s)']
            fixation_duration += (end_time - start_time)
            is_fixation_active = False  # 固視が終わったためリセット
    
    # データが途中で終わる可能性に備えて、最後に終了点がない場合を処理
    if is_fixation_active and start_time is not None:
        # 最後のレコードのタイムスタンプを終了点として計算
        end_time = df.iloc[-1]['Recording timestamp (s)']
        fixation_duration += (end_time - start_time)
    
    return round(fixation_duration * 1000, 1)

# 各ウィンドウの固視時間を計算してデータフレームに格納（高速版）
features['fixation_duration'] = [calculate_fixation_duration(window) for window in windows]

# 各ウィンドウの固視時間の特徴量を表示（確認用）
#for i in range(len(features)):
#    print(f"ウィンドウ {i+1} の固視時間: {features.at[i, 'fixation_duration']} ミリ秒")

固視回数 (FC) ：あるインスタンスにおけるすべての固視の回数

In [81]:
# 固視回数を計算する関数を定義
def calculate_fixation_count(df):
    fixation_count = 0
    previous_event_type = None
    
    for _, row in df.iterrows():
        current_event_type = row['Eye movement type']
        
        # 'Fixation' の場合のみ処理
        if current_event_type == 'Fixation':
            # 前のイベントが 'Fixation' でない場合、または異なる固視イベントの場合にカウント
            if previous_event_type != 'Fixation':
                fixation_count += 1
        
        # 現在のイベントタイプを次のループ用に保存
        previous_event_type = current_event_type
    
    return fixation_count

# 各ウィンドウの固視時間を計算してデータフレームに格納（高速版）
features['fixation_count'] = [calculate_fixation_count(window) for window in windows]

# 各ウィンドウの固視回数を表示（確認用）
#for i in range(len(features)):
#    print(f"ウィンドウ {i+1} の固視回数: {features.at[i, 'fixation_count']}")

固視分散 (FDI) ：インスタンス内の全固視座標の平均と各固視座標との距離の二乗平均の平方根

In [82]:
import numpy as np

# 固視分散を計算する関数を定義
def calculate_fixation_dispersion(df):
    
    # 固視座標を集計 (サッケードなど異なるイベントを挟んでも別として処理)
    fixation_points = []
    previous_event_type = None
    
    for _, row in df.iterrows():
        current_event_type = row['Eye movement type']
        
        # 固視イベントが連続していない場合のみ追加
        if current_event_type == 'Fixation' and previous_event_type != 'Fixation':
            fixation_points.append((row['Fixation point X'], row['Fixation point Y']))
        
        # イベントタイプを記録
        previous_event_type = current_event_type
    
    # 固視点がない場合は0を返す
    if len(fixation_points) == 0:
        return 0.0
    
    # X, Y の座標を別々に抽出
    fixation_points = np.array(fixation_points)
    fixation_x = fixation_points[:, 0]
    fixation_y = fixation_points[:, 1]
    
    # 固視座標の平均を計算
    mean_x = np.mean(fixation_x)
    mean_y = np.mean(fixation_y)
    
    # 各固視座標と平均座標との距離の二乗を計算
    squared_distances = (fixation_x - mean_x) ** 2 + (fixation_y - mean_y) ** 2
    
    # 固視分散を RMSD として計算
    fixation_dispersion = np.mean(np.sqrt(squared_distances))
    
    return round(fixation_dispersion, 1)

# 各ウィンドウの固視時間を計算してデータフレームに格納（高速版）
features['fixation_dispersion'] = [calculate_fixation_dispersion(window) for window in windows]

# 各ウィンドウの固視分散の特徴量を表示（確認用）
#for i in range(len(features)):
#    print(f"ウィンドウ {i+1} の固視分散: {features.at[i, 'fixation_dispersion']}")

サッケード時間 [ms] (SD)：インスタンス内のあるサッケードにかかった時間  

In [83]:
# サッケード時間の特徴量を計算する関数を定義
def calculate_saccade_duration(df):
    saccade_duration = 0
    is_saccade_active = False
    start_time = None
    
    for _, row in df.iterrows():
        event_type = row['Eye movement type']
        
        if event_type == 'Saccade':
            # サッケードが開始した場合、開始時間を記録
            if not is_saccade_active:
                start_time = row['Recording timestamp (s)']
                is_saccade_active = True
                
        elif event_type == 'Fixation' and is_saccade_active:
            # サッケード終了時に終了時間を記録し、差分を計算
            end_time = row['Recording timestamp (s)']
            saccade_duration += (end_time - start_time)
            is_saccade_active = False  # サッケードが終わったためリセット
    
    # データが途中で終わる可能性に備えて、最後に終了点がない場合を処理
    if is_saccade_active and start_time is not None:
        # 最後のレコードのタイムスタンプを終了点として計算
        end_time = df.iloc[-1]['Recording timestamp (s)']
        saccade_duration += (end_time - start_time)
    
    return round(saccade_duration * 1000, 1)

# 各ウィンドウの固視時間を計算してデータフレームに格納（高速版）
features['saccade_duration'] = [calculate_saccade_duration(window) for window in windows]

# 各ウィンドウのサッケード時間の特徴量を表示（確認用）
#for i in range(len(features)):
#    print(f"ウィンドウ {i+1} のサッケード時間: {features.at[i, 'saccade_duration']} ミリ秒")

サッケード回数 (SC) ：あるインスタンスにおけるすべてのサッケードの回数  

In [84]:
# サッケード回数を計算する関数を定義
def calculate_saccade_count(df):
    saccade_count = 0
    is_saccade_active = False
    start_time = None
    
    for _, row in df.iterrows():
        event_type = row['Eye movement type']
        
        if event_type == 'Saccade':
            if not is_saccade_active:
                # サッケードが開始した場合、開始時間を記録し、カウントを増やす
                start_time = row['Recording timestamp (s)']
                saccade_count += 1
                is_saccade_active = True
                
        elif event_type == 'Fixation' and is_saccade_active:
            # サッケード終了時に終了時間を記録し、サッケードが終了したとみなす
            end_time = row['Recording timestamp (s)']
            is_saccade_active = False  # サッケードが終わったためリセット
    
    # データが途中で終わる可能性に備えて、最後に終了点がない場合を処理
    if is_saccade_active and start_time is not None:
        end_time = df.iloc[-1]['Recording timestamp (s)']
        # 最後のサッケードはカウントに含まれるので、特に追加の処理は不要

    return saccade_count

# 各ウィンドウの固視時間を計算してデータフレームに格納（高速版）
features['saccade_count'] = [calculate_saccade_count(window) for window in windows]

# 各ウィンドウのサッケード回数の特徴量を表示（確認用）
#for i in range(len(features)):
#    print(f"ウィンドウ {i+1} のサッケード回数: {features.at[i, 'saccade_count']}")

サッケード振幅 [pixel] (SA)：2つの固視点間のピクセル数  

In [85]:
# サッケード振幅を計算する関数を定義
def calculate_saccade_amplitude(df):
    saccade_amplitude = 0
    is_saccade_active = False
    start_point = None
    previous_fixation_point = None
    
    for _, row in df.iterrows():
        event_type = row['Eye movement type']
        
        if event_type == 'Fixation'and not is_saccade_active:
            # 固視イベントが新しい場合、前の固視点として記録
            previous_fixation_point = (row['Fixation point X'], row['Fixation point Y'])
        
        elif event_type == 'Saccade':
            # サッケードが開始した場合、直前の固視点を記録
            if not is_saccade_active and previous_fixation_point is not None:
                start_point = previous_fixation_point
                is_saccade_active = True
                
        elif event_type == 'Fixation' and is_saccade_active:
            # サッケード終了時に終了点を記録し、振幅を計算
            if previous_fixation_point is not None:
                end_point = (row['Fixation point X'], row['Fixation point Y'])
                if start_point:
                    # サッケードの振幅を計算
                    dx = end_point[0] - start_point[0]
                    dy = end_point[1] - start_point[1]
                    amplitude = np.sqrt(dx**2 + dy**2)
                    saccade_amplitude += amplitude
            is_saccade_active = False  # サッケードが終わったためリセット
    
    # データが途中で終わる可能性に備えて、最後に終了点がない場合を処理
    #if is_saccade_active and start_point is not None and previous_fixation_point is not None:
        # 最後のサッケードの終了点は最後のレコードから計算
    #    end_point = (df.iloc[-1]['Gaze point X'], df.iloc[-1]['Gaze point Y'])
    #    dx = end_point[0] - start_point[0]
    #    dy = end_point[1] - start_point[1]
    #    amplitude = np.sqrt(dx**2 + dy**2)
    #    saccade_amplitude += amplitude

    return round(saccade_amplitude, 1)

# 各ウィンドウの固視時間を計算してデータフレームに格納（高速版）
features['saccade_amplitude'] = [calculate_saccade_amplitude(window) for window in windows]

# 各ウィンドウのサッケード振幅の特徴量を表示（確認用）
#for i in range(len(features)):
#    print(f"ウィンドウ {i+1} のサッケード振幅: {features.at[i, 'saccade_amplitude']} ピクセル")

サッケード速度 [pixel/ms] (SV) ：サッケード振幅/サッケード時間  

In [86]:
# サッケード速度を計算する関数を定義
def calculate_saccade_velocity(saccade_amplitude, saccade_duration):
    # サッケード時間が0の場合は速度を0とする
    if saccade_duration == 0:
        return 0.0
    # サッケード速度 = 振幅 / 時間
    saccade_velocity = saccade_amplitude / saccade_duration
    return round(saccade_velocity, 2)

# 各ウィンドウのサッケード振幅、時間、速度を計算してデータフレームに格納
for i in range(len(features)):
    amplitude = features.at[i, 'saccade_amplitude']  # サッケード振幅
    duration = features.at[i, 'saccade_duration']    # サッケード時間
    features.at[i, 'saccade_velocity'] = calculate_saccade_velocity(amplitude, duration)  # 変更箇所

# 各ウィンドウのサッケード速度の特徴量を表示（確認用）
#for i in range(len(features)):
#    print(f"ウィンドウ {i+1}: サッケード速度: {features.at[i, 'saccade_velocity']:.2f} ピクセル/ms")

相対サッケード角度分布 (RSA) ：2つのサッケードの間の角度  

In [87]:
# 2つのベクトル間の角度を計算する関数
def calculate_angle_between_vectors(v1, v2):
    # ベクトルの内積
    dot_product = np.dot(v1, v2)
    # ベクトルの大きさの積
    magnitude_product = np.linalg.norm(v1) * np.linalg.norm(v2)
    
    if magnitude_product == 0:
        return 0.0
    
    # コサインの逆関数で角度を求める (ラジアン)
    cos_angle = dot_product / magnitude_product
    cos_angle = np.clip(cos_angle, -1.0, 1.0)  # 数値誤差による範囲外を防ぐ
    angle = np.arccos(cos_angle)
    
    # ラジアンから度に変換
    angle_degrees = np.degrees(angle)
    
    return angle_degrees

# 相対サッカード角度分布 (RSA) を計算する関数
def calculate_relative_saccade_angle(df):
    fixation_points = []
    previous_event_type = None
    
    # 固視イベントの座標を収集、同じ固視でもサッケードで区切られていれば別とみなす
    for _, row in df.iterrows():
        current_event_type = row['Eye movement type']
        
        if current_event_type == 'Fixation':
            current_fixation = (row['Fixation point X'], row['Fixation point Y'])
            
            # サッケードで区切られた場合は同じ座標でも別の固視として扱う
            if previous_event_type != 'Fixation':
                fixation_points.append(current_fixation)
        
        # 前回のイベントタイプを記録
        previous_event_type = current_event_type
    
    # 固視点が3つ未満の場合は計算できないので0を返す
    if len(fixation_points) < 3:
        return 0.0
    
    rsa_values = []
    
    # 3つの連続した固視点を用いて角度を計算
    for i in range(len(fixation_points) - 2):
        # 2つ目から1つ目のベクトル
        v1 = np.array(fixation_points[i]) - np.array(fixation_points[i+1])
        # 2つ目から3つ目のベクトル
        v2 = np.array(fixation_points[i+2]) - np.array(fixation_points[i+1])
        
        # 2つのベクトルの間の角度を計算
        angle = calculate_angle_between_vectors(v1, v2)
        rsa_values.append(angle)
    
    # 平均相対サッカード角度を返す
    return round(np.mean(rsa_values), 1)

# 各ウィンドウの固視時間を計算してデータフレームに格納（高速版）
features['relative_saccade_angle'] = [calculate_relative_saccade_angle(window) for window in windows]

# 各ウィンドウのRSAの特徴量を表示（確認用）
#for i in range(len(features)):
#    print(f"ウィンドウ {i+1}: 相対サッケード角度分布 (RSA): {features.at[i, 'relative_saccade_angle']:.1f} 度")

絶対サッケード角度分布 (ASA)：2つの固視点間の線分と x 軸の間の角度  

In [88]:
# 2つの固視点間の線分とx軸の間の角度を計算する関数
def calculate_absolute_saccade_angle(df):
    fixation_points = []
    previous_event_type = None
    
    # 固視イベントの座標を収集、同じ固視でもサッケードで区切られていれば別とみなす
    for _, row in df.iterrows():
        current_event_type = row['Eye movement type']
        
        if current_event_type == 'Fixation':
            current_fixation = (row['Fixation point X'], row['Fixation point Y'])
            
            # サッケードで区切られた場合は同じ座標でも別の固視として扱う
            if previous_event_type != 'Fixation':
                fixation_points.append(current_fixation)
        
        # 前回のイベントタイプを記録
        previous_event_type = current_event_type
    
    # 固視点が2つ未満の場合は計算できないので0を返す
    if len(fixation_points) < 2:
        return 0.0
        
    # 2つの連続した固視点を用いて角度を計算
    for i in range(len(fixation_points) - 1):
        # 固視点のX, Y座標の差分を計算
        delta_x = fixation_points[i][0] - fixation_points[i+1][0]
        delta_y = fixation_points[i][1] - fixation_points[i+1][1]
        
        # Y座標差とX座標差を用いて角度を計算 (ラジアン)
        angle_radians = np.arctan2(delta_y, delta_x)
        
        # ラジアンから度に変換
        angle_degrees = np.degrees(angle_radians)
            
    # 平均絶対サッケード角度を返す
    return round(np.mean(angle_degrees), 2)

# 各ウィンドウの固視時間を計算してデータフレームに格納（高速版）
features['absolute_saccade_angle'] = [calculate_absolute_saccade_angle(window) for window in windows]

# 各ウィンドウのASAの特徴量を表示（確認用）
#for i in range(len(features)):
#    print(f"ウィンドウ {i+1}: 絶対サッケード角度分布 (ASA): {features.at[i, 'absolute_saccade_angle']} 度")

垂直サッケード割合 (HSR) ：x 軸の上下に 30 度以内の角度を持つサッケードの割合  

In [89]:
# 水平サッケード割合 (HSR) を計算する関数
def calculate_horizontal_saccade_ratio(df):
    fixation_points = []
    previous_event_type = None
    
    # 固視イベントの座標を収集
    for _, row in df.iterrows():
        current_event_type = row['Eye movement type']
        
        if current_event_type == 'Fixation':
            current_fixation = (row['Fixation point X'], row['Fixation point Y'])
            
            # サッケードで区切られた場合は同じ座標でも別の固視として扱う
            if previous_event_type != 'Fixation':
                fixation_points.append(current_fixation)
        
        # 前回のイベントタイプを記録
        previous_event_type = current_event_type
    
    # 固視点が2つ未満の場合は計算できないので0を返す
    if len(fixation_points) < 2:
        return 0.0
    
    horizontal_saccades = 0
    total_saccades = 0
    
    # 2つの連続した固視点を用いて角度を計算
    for i in range(len(fixation_points) - 1):
        # 固視点のX, Y座標の差分を計算
        delta_x = fixation_points[i][0] - fixation_points[i+1][0]
        delta_y = fixation_points[i][1] - fixation_points[i+1][1]
        
        # Y座標差とX座標差を用いて角度を計算 (ラジアン)
        angle_radians = np.arctan2(delta_y, delta_x)
        
        # ラジアンから度に変換
        angle_degrees = np.degrees(angle_radians)
        
        # サッケード角度が-120度〜-60度の範囲内であればカウント
        if -120 <= angle_degrees <= -60:
            horizontal_saccades += 1
        
        total_saccades += 1
    
    # サッケードがない場合は0を返す
    if total_saccades == 0:
        return 0.0
    
    # 水平サッケードの割合を計算
    hsr = (horizontal_saccades / total_saccades) * 100
    
    return round(hsr, 2)

# 各ウィンドウの固視時間を計算してデータフレームに格納（高速版）
features['horizontal_saccade_ratio'] = [calculate_horizontal_saccade_ratio(window) for window in windows]

# 各ウィンドウのHSRの特徴量を表示（確認用）
#for i in range(len(features)):
#    print(f"ウィンドウ {i+1}: 垂直サッケード割合 (HSR): {features.at[i, 'horizontal_saccade_ratio']}%")

固視時間/サッケード時間 (FSR) ：固視時間とサッカード時間との比

In [90]:
# 固視時間/サッケード時間 (FSR) を計算する関数
def calculate_fixation_saccade_ratio(fixation_duration, saccade_duration):
    # サッケード時間が0の場合はFSRを0とする
    if saccade_duration == 0:
        return 0.0
    # 固視時間 / サッケード時間 の比率を計算
    fsr = fixation_duration / saccade_duration
    return round(fsr, 2)

# 各ウィンドウの固視時間、サッカード時間、FSRを計算してデータフレームに格納
for i in range(len(features)):
    fixation_duration = features.at[i, 'fixation_duration']  # 固視時間
    saccade_duration = features.at[i, 'saccade_duration']    # サッカード時間
    features.at[i, 'fixation_saccade_ratio'] = calculate_fixation_saccade_ratio(fixation_duration, saccade_duration)

# 各ウィンドウの固視時間/サッケード時間 (FSR) の特徴量を表示（確認用）
#for i in range(len(features)):
#    print(f"ウィンドウ {i+1}: 固視時間/サッケード時間 (FSR): {features.at[i, 'fixation_saccade_ratio']:.2f}")

左目の瞳孔径 [mm]：左目の瞳孔の大きさ  
右目の瞳孔径 [mm]：右目の瞳孔の大きさ

In [91]:
# 左右の瞳孔径を計算する関数（高速化）
def calculate_pupil_diameter(df):
    left_mean = df['Pupil diameter left'].mean(skipna=True)
    right_mean = df['Pupil diameter right'].mean(skipna=True)
    left = round(left_mean, 2) if pd.notna(left_mean) else 0.0
    right = round(right_mean, 2) if pd.notna(right_mean) else 0.0
    return left, right

# 各ウィンドウの瞳孔径を計算してデータフレームに格納（高速化）
features[['left_pupil_diameter', 'right_pupil_diameter']] = [
    calculate_pupil_diameter(window) for window in windows
]

# 各ウィンドウの瞳孔径の特徴量を表示（確認用）
#for i in range(len(features)):
#    print(f"ウィンドウ {i+1}: 瞳孔径   左: {features.at[i, 'left_pupil_diameter']}mm   右: {features.at[i, 'right_pupil_diameter']}mm")

### 打鍵特徴量計算

打鍵間隔平均 (keystroke_interval_mean): 連続する打鍵の間隔の平均  
打鍵間隔標準偏差 (keystroke_interval_std): 打鍵間隔のばらつき  
打鍵の合計数 (keystroke_count): ウィンドウ内の打鍵の総数  
平均打鍵速度 (average_keystroke_speed): 打鍵間隔の逆数の平均  
打鍵速度の標準偏差 (keystroke_speed_std): 打鍵速度の標準偏差  
最大打鍵速度 (keystroke_max_speed): 最速の打鍵間隔（最短時間の間隔）  
最小打鍵速度 (keystroke_min_speed): 最長の打鍵間隔  
削除回数 (delete_key_count): ウィンドウ内で「Backspace」キーが押された回数  
削除頻度 (delete_key_frequency): 削除回数/ウィンドウ内の総打鍵数  

In [92]:
# 各ウィンドウに対応する特徴量を計算して追加
for i, window_data in enumerate(windows):
    # ウィンドウ内の打鍵データを抽出
    window_key_data = df_cleaned[df_cleaned['Recording timestamp (s)'].between(
        window_data['Recording timestamp (s)'].min(), 
        window_data['Recording timestamp (s)'].max())]

    # キー列に情報が格納されている回数をカウント
    keystroke_count = len(window_key_data)

    # 打鍵間隔の計算
    keystroke_intervals = window_key_data['Recording timestamp (s)'].diff().dropna()
    keystroke_interval_mean = keystroke_intervals.mean() if not keystroke_intervals.empty else 0
    keystroke_interval_std = keystroke_intervals.std() if len(keystroke_intervals) > 1 else 0

    # 打鍵速度の計算（例: 時間単位で打鍵回数を計算）
    if keystroke_count > 1:
        non_zero_intervals = keystroke_intervals[keystroke_intervals > 0]
        keystroke_speeds = 1 / non_zero_intervals  # 速度 = 1 / 間隔
        keystroke_max_speed = keystroke_speeds.max() if not keystroke_speeds.empty else 0
        keystroke_min_speed = keystroke_speeds.min() if not keystroke_speeds.empty else 0
        average_keystroke_speed = keystroke_speeds.mean() if not keystroke_speeds.empty else 0
        keystroke_speed_std = keystroke_speeds.std() if len(non_zero_intervals) > 1 else 0
    else:
        keystroke_max_speed = 0
        keystroke_min_speed = 0
        average_keystroke_speed = 0
        keystroke_speed_std = 0

    # 削除キー（Key.backspace）のカウントと頻度の計算
    delete_key_count = window_key_data[window_key_data['Key'] == 'Key.backspace'].shape[0]
    delete_key_frequency = delete_key_count / keystroke_count if keystroke_count > 0 else 0

    # 特徴量をfeaturesデータフレームの該当する行に追加
    features.at[i, 'keystroke_interval_mean'] = keystroke_interval_mean
    features.at[i, 'keystroke_interval_std'] = keystroke_interval_std
    features.at[i, 'keystroke_count'] = keystroke_count
    features.at[i, 'keystroke_max_speed'] = keystroke_max_speed
    features.at[i, 'keystroke_min_speed'] = keystroke_min_speed
    features.at[i, 'average_keystroke_speed'] = average_keystroke_speed
    features.at[i, 'keystroke_speed_std'] = keystroke_speed_std
    features.at[i, 'delete_key_count'] = delete_key_count
    features.at[i, 'delete_key_frequency'] = delete_key_frequency


### 圧力特徴量計算

各ウィンドウ毎に圧力分布の平均を計算

In [93]:
feature_sit = []

for window in windows:
    # vol1からvol29までの列の平均値を計算
    avg_values = window[[f'vol{i}' for i in range(1, 30)]].mean(axis=0, skipna=True)  # 各列の平均を計算
    
    # 計算した平均値をデータフレームとして追加
    feature_sit.append(avg_values)

# feature_sitを一つのデータフレームに結合
feature_sit = pd.DataFrame(feature_sit).reset_index(drop=True)

# 結果を表示
print(feature_sit)

          vol1      vol2      vol3      vol4      vol5      vol6      vol7  \
0    -0.713974 -0.270673  0.145569  0.020106 -0.666574 -0.816168 -0.957573   
1    -0.584615 -0.302017  0.227118  0.066729 -0.684237 -0.558018 -0.759378   
2    -0.395698 -0.326569  0.294890  0.138947 -0.700229 -0.212575 -0.510967   
3    -0.028113  0.025537  0.181459  0.078277 -0.683531  0.037475 -0.331001   
4     0.568657  0.766108  0.132112 -0.014753 -0.677716  0.487153  0.086750   
...        ...       ...       ...       ...       ...       ...       ...   
3772  1.225900  1.565879 -0.041200  0.579163  1.018526  0.612241  0.429575   
3773  1.168534  1.545366 -0.041618  0.546769  1.026336  0.524258  0.445087   
3774  1.149052  1.563009 -0.041685  0.528295  1.058804  0.430968  0.532485   
3775  1.166907  1.624037 -0.041848  0.504809  1.069787  0.481467  0.638626   
3776  1.189458  1.602142 -0.041820  0.384360  0.916480  0.527179  0.643709   

          vol8      vol9     vol10  ...     vol20     vol21    

vol1~vol25，vol26~vol29を正規化

In [94]:
# 座面（vol1~vol25）の正規化
seat_columns = [f'vol{i}' for i in range(1, 26)]
seat_min = feature_sit[seat_columns].min().min()  # 座面全体の最小値
seat_max = feature_sit[seat_columns].max().max()  # 座面全体の最大値

# 座面の各列を同じ範囲で正規化
feature_sit[seat_columns] = (feature_sit[seat_columns] - seat_min) / (seat_max - seat_min)

# 背もたれ（vol26~vol29）の正規化
backrest_columns = [f'vol{i}' for i in range(26, 30)]
backrest_min = feature_sit[backrest_columns].min().min()  # 背もたれ全体の最小値
backrest_max = feature_sit[backrest_columns].max().max()  # 背もたれ全体の最大値

# 背もたれの各列を同じ範囲で正規化
feature_sit[backrest_columns] = (feature_sit[backrest_columns] - backrest_min) / (backrest_max - backrest_min)

# 結果を表示
print(feature_sit)

          vol1      vol2      vol3      vol4      vol5      vol6      vol7  \
0     0.392192  0.448597  0.501558  0.485594  0.398223  0.379189  0.361197   
1     0.408651  0.444608  0.511934  0.491527  0.395976  0.412036  0.386415   
2     0.432689  0.441484  0.520557  0.500716  0.393941  0.455989  0.418022   
3     0.479459  0.486286  0.506125  0.492996  0.396066  0.487805  0.440921   
4     0.555391  0.580514  0.499846  0.481159  0.396805  0.545020  0.494074   
...        ...       ...       ...       ...       ...       ...       ...   
3772  0.639017  0.682275  0.477794  0.556728  0.612631  0.560936  0.537694   
3773  0.631718  0.679665  0.477741  0.552606  0.613625  0.549742  0.539668   
3774  0.629239  0.681910  0.477732  0.550255  0.617756  0.537872  0.550788   
3775  0.631511  0.689675  0.477712  0.547267  0.619153  0.544297  0.564293   
3776  0.634380  0.686889  0.477715  0.531941  0.599647  0.550113  0.564940   

          vol8      vol9     vol10  ...     vol20     vol21    

6×5のヒートマップを作成

In [95]:
"""import seaborn as sns
import matplotlib.pyplot as plt
import os
import numpy as np

# ヒートマップ画像を保存するディレクトリを作成
output_dir = 'heatmaps'
os.makedirs(output_dir, exist_ok=True)

# feature_sitの行数分だけヒートマップを作成
for i in range(len(feature_sit)):
    # 各行のデータを取得
    row_values = feature_sit.iloc[i, :].values  # 行のデータを取得
    row_values = np.append(row_values, 0)  # 30個目のデータとして0を追加（オプション）

    # 5×6のヒートマップ用にデータを整形
    heatmap_data = row_values.reshape(5, 6)  # 5×6に変形

    # ヒートマップを作成
    plt.figure(figsize=(10, 6))
    sns.heatmap(heatmap_data, annot=True, cmap='Greens', cbar_kws={'label': 'Mean Value'})

    # X軸ラベル
    plt.xticks(ticks=np.arange(6) + 0.5, labels=[f'vol{i+1}' for i in range(6)], rotation=45)
    # Y軸ラベル
    plt.yticks(ticks=np.arange(5) + 0.5, labels=[f'Group {j+1}' for j in range(5)], rotation=0)

    plt.title(f'Heatmap for Feature Set {i+1}')
    plt.xlabel('Sensors (vol1 to vol29)')
    plt.ylabel('Groups')

    # ヒートマップを画像ファイルとして保存
    plt.savefig(f'{output_dir}/heatmap_feature_{i+1}.png')
    plt.close()  # プロットを閉じてメモリを解放

print(f'Created {len(feature_sit)} heatmaps in the "{output_dir}" directory.')"""


'import seaborn as sns\nimport matplotlib.pyplot as plt\nimport os\nimport numpy as np\n\n# ヒートマップ画像を保存するディレクトリを作成\noutput_dir = \'heatmaps\'\nos.makedirs(output_dir, exist_ok=True)\n\n# feature_sitの行数分だけヒートマップを作成\nfor i in range(len(feature_sit)):\n    # 各行のデータを取得\n    row_values = feature_sit.iloc[i, :].values  # 行のデータを取得\n    row_values = np.append(row_values, 0)  # 30個目のデータとして0を追加（オプション）\n\n    # 5×6のヒートマップ用にデータを整形\n    heatmap_data = row_values.reshape(5, 6)  # 5×6に変形\n\n    # ヒートマップを作成\n    plt.figure(figsize=(10, 6))\n    sns.heatmap(heatmap_data, annot=True, cmap=\'Greens\', cbar_kws={\'label\': \'Mean Value\'})\n\n    # X軸ラベル\n    plt.xticks(ticks=np.arange(6) + 0.5, labels=[f\'vol{i+1}\' for i in range(6)], rotation=45)\n    # Y軸ラベル\n    plt.yticks(ticks=np.arange(5) + 0.5, labels=[f\'Group {j+1}\' for j in range(5)], rotation=0)\n\n    plt.title(f\'Heatmap for Feature Set {i+1}\')\n    plt.xlabel(\'Sensors (vol1 to vol29)\')\n    plt.ylabel(\'Groups\')\n\n    # ヒートマップを画像ファイル

ヒートマップをCNNで特徴量に変換

In [96]:
import tensorflow as tf
from tensorflow.keras import layers, models
import numpy as np
import pandas as pd

# 乱数シードの固定
SEED = 0
np.random.seed(SEED)
tf.random.set_seed(SEED)

# 1. データ準備
# 各行を5×6の形に整形してCNNに適した入力形式にする
data = []
for i in range(len(feature_sit)):
    row_values = np.append(feature_sit.iloc[i, :].values, 0)  # 30個目に0を追加して30個に
    data.append(row_values.reshape(5, 6))  # 5×6に変形

# CNN用の入力データ形式に変換
data = np.array(data)
data = np.expand_dims(data, axis=-1)  # チャネル次元を追加 (サンプル数, 5, 6, 1)

# 2. Functional APIを使ってCNNモデルを構築
inputs = layers.Input(shape=(5, 6, 1))
x = layers.Conv2D(16, (2, 2), activation='relu')(inputs)
x = layers.MaxPooling2D((2, 2))(x)
x = layers.Conv2D(32, (2, 2), activation='relu')(x)
x = layers.Flatten()(x)
x = layers.Dense(64, activation='relu')(x)
output_features = layers.Dense(10)(x)  # 特徴量として抽出する10次元のベクトル

# モデルを構築
model = models.Model(inputs=inputs, outputs=output_features)

# **ランダム性をなくすため、モデルの重みを固定**
model.compile(optimizer='adam', loss='mse')  
model.set_weights([np.random.RandomState(SEED).randn(*w.shape) for w in model.get_weights()])

# 特徴量の抽出
extracted_features = model.predict(data)

# 特徴量をデータフレームに変換して 'sit1' 〜 'sit10' 列として features に追加
sit_features_df = pd.DataFrame(extracted_features, columns=[f'sit{i+1}' for i in range(10)])
features.loc[:, [f'sit{i+1}' for i in range(10)]] = sit_features_df.values

# 特徴量データフレームの内容を確認
print(features.head())

119/119 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step
   Participant_ID  Gender_ID  Task_ID  Block_ID  Confusion  Difficulty  \
0               0          1       51         1        0.0         0.0   
1               0          1       51         1        0.0         0.0   
2               0          1       51         1        0.0         0.0   
3               0          1       51         1        0.0         0.0   
4               0          1       51         1        0.0         0.0   

   fixation_duration  fixation_count  fixation_dispersion  saccade_duration  \
0            24149.2             115                189.0            7109.8   
1            22481.6             112                243.1            8953.1   
2            21676.6             104                346.9            9177.6   
3            24480.3             112                327.8            6462.3   
4            23933.4             111                281.8            6727.8   

   ...        sit1        sit2       si

seat_net_pressure_mean    ：座面の平均圧力  
seat_net_pressure_std     ：座面圧力のばらつき  
back_net_pressure_mean    ：背もたれの平均圧力  
back_net_pressure_std     ：背もたれ圧力のばらつき  
seat_pressure_change_mean ：座面の平均活動量（微小な動き）  
back_pressure_change_mean ：背もたれの平均活動量  
seat_coverage_mean        ：座面の接触割合（coverage）    
back_coverage_mean      　：背もたれの接触割合  
seat_back_pressure_ratio　：seat_mean / back_mean  
seat_back_pressure_diff　 ：seat_mean - back_mean  
seat_back_activity_ratio  ：seat_activity / back_activity

In [97]:
import numpy as np
import pandas as pd

pressure_feature_list = []  # ← 別名に

epsilon = 1e-6  # ゼロ除算回避

for w in windows:
    seat_data = w[[f'vol{i}' for i in range(1, 26)]]
    back_data = w[[f'vol{i}' for i in range(26, 30)]]
    
    # 各時刻の座面・背もたれの平均圧力
    seat_net = seat_data.mean(axis=1)
    back_net = back_data.mean(axis=1)
    
    # ウィンドウ統計
    seat_mean = seat_net.mean()
    seat_std = seat_net.std()
    back_mean = back_net.mean()
    back_std = back_net.std()
    
    # 圧力変化量（activity）
    seat_diff = seat_net.diff().abs().dropna()
    back_diff = back_net.diff().abs().dropna()
    seat_change = seat_diff.mean() if not seat_diff.empty else 0.0
    back_change = back_diff.mean() if not back_diff.empty else 0.0
    
    # coverage
    seat_cov = (seat_data > 0.1).mean(axis=1).mean()
    back_cov = (back_data > 0.1).mean(axis=1).mean()
    
    # seat vs back の関係
    seat_back_ratio = seat_mean / (back_mean + epsilon)
    seat_back_diff = seat_mean - back_mean
    seat_back_activity_ratio = seat_change / (back_change + epsilon)
    
    # 結果を辞書でまとめてリストに
    pressure_feature_list.append({
        'seat_net_pressure_mean': seat_mean,
        'seat_net_pressure_std': seat_std,
        'back_net_pressure_mean': back_mean,
        'back_net_pressure_std': back_std,
        'seat_pressure_change_mean': seat_change,
        'back_pressure_change_mean': back_change,
        'seat_coverage_mean': seat_cov,
        'back_coverage_mean': back_cov,
        'seat_back_pressure_ratio': seat_back_ratio,
        'seat_back_pressure_diff': seat_back_diff,
        'seat_back_activity_ratio': seat_back_activity_ratio
    })
    
# DataFrameに変換して統合
pressure_features = pd.DataFrame(pressure_feature_list)
features = pd.DataFrame(features)
features = pd.concat([features.reset_index(drop=True), pressure_features.reset_index(drop=True)], axis=1)

### 欠損値の補完＆特徴量の保存

In [98]:
import os

# 欠損がある列を抽出
columns_with_nan = features.columns[features.isna().any()].tolist()

# 欠損値を 0 で補完
features.fillna(0, inplace=True)

# 補完された列名を表示
print("欠損を0で補完した列名：")
for col in columns_with_nan:
    print(col)

# 計算した全特徴量名を表示
# print("\n計算した全特徴量名：")
# for col in features.columns:
#     print(col)

# 補完後に保存
features.to_csv('features.csv', index=False)

save_dir = r"C:\Users\funah\Desktop\実験関連\実験データ\feature\12_feature_non_delete"
save_path = os.path.join(save_dir, f"features_ws{window_size}.csv")

# 保存
features.to_csv(save_path, index=False)

# 保存したファイル名を出力
print(f"保存しました: {save_path}")

欠損を0で補完した列名：
保存しました: C:\Users\funah\Desktop\実験関連\実験データ\feature\12_feature_non_delete\features_ws32.csv
